# Offshore Wind Resource Performance Data 

### Implementation

In [ ]:
# Imports 
import pandas as pd 
import numpy as np 

In [ ]:
# Read offshore resource performance file 
import h5py

filepath = "inputs/wind-ofs-limited_county.h5"

with h5py.File(filepath, "r") as f:
    cf = f["cf"][:]          # NumPy array
    index = f["index"][:]    # time labels
    columns = f["columns"][:]  # county labels


In [ ]:
# Fix data types, check output
index = index.astype(str)
columns = columns.astype(str)
df = pd.DataFrame(cf, index=index, columns=columns)

print(df.head())
print(df.shape)
print(df.index[:5])


   6_p01003  14_p01003  6_p01097  14_p01097  8_p06015  10_p06015  8_p06023  \
1  0.195312   0.119629  0.120911   0.044403    0.0021        0.0  0.000900   
2  0.269287   0.135254  0.268799   0.189941    0.0015        0.0  0.003099   
3  0.336426   0.166626  0.408691   0.239380    0.0037        0.0  0.011002   
4  0.397217   0.220337  0.374023   0.406250    0.0004        0.0  0.013298   
5  0.388916   0.268311  0.378906   0.401855    0.0000        0.0  0.021500   

   9_p06023  10_p06023  11_p06023  ...  13_p55003  12_p55029  13_p55059  \
1    0.0000        0.0        0.0  ...   0.990234   0.996094   1.000000   
2    0.0000        0.0        0.0  ...   0.933594   0.958984   1.000000   
3    0.0000        0.0        0.0  ...   0.999512   0.580566   0.978027   
4    0.0000        0.0        0.0  ...   0.997559   0.673340   1.000000   
5    0.0014        0.0        0.0  ...   0.999512   0.835449   0.885742   

   12_p55061  12_p55071  4_p55079  12_p55079  12_p55101  13_p55101  12_p55117  


C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# Extract metadata from column names, pack into dataframe 
cols = df.columns.to_series()

meta = cols.str.extract(
    r"(?P<class>\d+)_p(?P<fips>\d+)"
)

meta["class"] = meta["class"].astype(int)
meta["fips"] = meta["fips"].astype(str).str.zfill(5)

df.columns = pd.MultiIndex.from_frame(meta)
df.columns.names = ["class", "fips"]

# Select average county-wide capacity factor (accross different resource classes)
county_cf = df.groupby(level="fips", axis=1).mean()


In [ ]:
# Get FIPS translator 
fips_translate_filename = "inputs/state_and_county_fips_master.csv"

fips_translator = pd.read_csv(fips_translate_filename, dtype={"fips": str})
fips_translator['fips'] = fips_translator['fips'].apply(lambda x: str(x).zfill(5))

In [ ]:
# Create a mapping from fips number to "State_County"
fips_map = fips_translator.set_index('fips').apply(lambda row: f"{row['state']}_{row['name']}", axis=1).to_dict()

new_columns = pd.MultiIndex.from_tuples(
    [(cls, fips_map.get(fips, fips)) for cls, fips in df.columns],
    names=df.columns.names
)

df.columns = new_columns



In [ ]:
# Check output
print(df.head())

class                6                 14               6                14  \
fips  AL_Baldwin County AL_Baldwin County AL_Mobile County AL_Mobile County   
1              0.195312          0.119629         0.120911         0.044403   
2              0.269287          0.135254         0.268799         0.189941   
3              0.336426          0.166626         0.408691         0.239380   
4              0.397217          0.220337         0.374023         0.406250   
5              0.388916          0.268311         0.378906         0.401855   

class                  8                   10                 8   \
fips  CA_Del Norte County CA_Del Norte County CA_Humboldt County   
1                  0.0021                 0.0           0.000900   
2                  0.0015                 0.0           0.003099   
3                  0.0037                 0.0           0.011002   
4                  0.0004                 0.0           0.013298   
5                  0.0000             

C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# Throw out all non-PJM resources 
PJM_states = ['PA', 'NJ', 'MD', 'DC', 'DE', 'IL', 'WV', 'VA', 'OH', 'IN', 'MI', 'KY', 'NC']  # added MI, KY, NC as commonly in PJM


# 2️⃣ Boolean mask for columns where the state is in PJM_states
mask = df.columns.get_level_values(1).str[:2].isin(PJM_states)

# 3️⃣ Filter columns
PJM_df = df.loc[:, mask]

PJM_frame = PJM_df.mean(axis=1).to_frame(name='pjm_avg')
PJM_frame.index = PJM_frame.index.astype(int)

PJM_frame['day'] = PJM_frame.index // 24


In [ ]:
# Import weather data for the appropriate periond and add to frame
weather = pd.read_csv("inputs/solar_weather_data.csv")
thi_series = weather['system_max_thi']

PJM_frame['thi'] = thi_series

PJM_frame['thi'] = thi_series

print(PJM_frame)


        pjm_avg   day        thi
1      0.877441     0  57.432200
2      0.906250     0  56.804000
3      0.929688     0  56.174000
4      0.956055     0  57.223400
5      0.964844     0  58.605589
...         ...   ...        ...
61316  0.639648  2554  62.255973
61317  0.673828  2554  62.817728
61318  0.703613  2554  60.487492
61319  0.755371  2554  56.683400
61320  0.822754  2555  53.083397

[61320 rows x 3 columns]


C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# Copy to separate dataframe (probably not necessary)
wind_performance_dataframe = PJM_frame.copy()
print(wind_performance_dataframe.head(24))

     pjm_avg  day        thi
1   0.877441    0  57.432200
2   0.906250    0  56.804000
3   0.929688    0  56.174000
4   0.956055    0  57.223400
5   0.964844    0  58.605589
6   0.970215    0  59.265292
7   0.963379    0  59.768344
8   0.951660    0  60.026889
9   0.963379    0  60.675020
10  0.979492    0  61.451061
11  0.956055    0  61.833970
12  0.958008    0  62.037908
13  0.913086    0  62.124884
14  0.902344    0  62.430400
15  0.893066    0  63.009694
16  0.879883    0  63.337932
17  0.860352    0  64.141077
18  0.838867    0  64.600890
19  0.811523    0  65.096168
20  0.798340    0  65.290886
21  0.767090    0  64.759673
22  0.761719    0  64.064048
23  0.764160    0  63.929366
24  0.750977    1  63.929366


C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# Get rid of date column, add datetime column and season column
wind_performance_dataframe['datetime'] = weather['date']
cols_to_keep = ['datetime', 'pjm_avg', 'thi']
wind_performance_dataframe = wind_performance_dataframe[cols_to_keep]

wind_performance_dataframe['datetime'] = pd.to_datetime(
    wind_performance_dataframe['datetime']
)

wind_performance_dataframe['season'] = np.where(
    wind_performance_dataframe['datetime'].dt.month.isin([10, 11, 12, 1, 2, 3]),
    'winter', 
    'summer'
)

wind_performance_dataframe['date'] = wind_performance_dataframe['datetime'].dt.date

In [ ]:
# Find peak daily THI values and bin them 
daily_thi = (
    wind_performance_dataframe.groupby(["date", "season"])["thi"]
      .agg(
          thi_extreme=lambda x: x.max() if x.name[1] == "summer" else x.min()
      )
      .reset_index()
)

bin_edges = np.arange(daily_thi["thi_extreme"].min() // 5 * 5,
                      daily_thi["thi_extreme"].max() + 5,
                      5)

daily_thi["thi_bin"] = pd.cut(
    daily_thi["thi_extreme"],
    bins=bin_edges
)


In [ ]:
# Grab daily 24-hour wind performance profiles 
daily_profiles = (
    wind_performance_dataframe.sort_values("datetime")
      .groupby("date")["pjm_avg"]
      .apply(lambda x: x.values if len(x) == 24 else None)
      .reset_index(name="wind_profile")
)
daily_profiles = daily_profiles.dropna()


In [ ]:
# Save in useful format
daily = daily_thi.merge(daily_profiles, on="date", how="inner")
daily.head()

,date,season,thi_extreme,thi_bin,wind_profile
0,2007-01-02,winter,44.173400,"(40.0, 45.0]","[0.751, 0.7207, 0.682, 0.66, 0.6416, 0.628, 0...."
1,2007-01-03,winter,37.693400,"(35.0, 40.0]","[0.509, 0.4946, 0.4812, 0.474, 0.4707, 0.4702,..."
2,2007-01-04,winter,49.753400,"(45.0, 50.0]","[0.737, 0.7334, 0.7266, 0.7256, 0.72, 0.7183, ..."
3,2007-01-05,winter,58.115310,"(55.0, 60.0]","[0.8257, 0.827, 0.8374, 0.8516, 0.843, 0.822, ..."
4,2007-01-06,winter,63.765006,"(60.0, 65.0]","[0.65, 0.6553, 0.6665, 0.6787, 0.701, 0.7046, ..."


In [ ]:
# Lengthen for save to CSV 
long = (
    daily
    .explode("wind_profile")
    .assign(hour=lambda x: x.groupby("date").cumcount())
)

long.to_csv("daily_offshore_profiles_long.csv", index=False)

print(long)

            date  season  thi_extreme       thi_bin wind_profile  hour
0     2007-01-02  winter      44.1734  (40.0, 45.0]     0.750977     0
0     2007-01-02  winter      44.1734  (40.0, 45.0]     0.720703     1
0     2007-01-02  winter      44.1734  (40.0, 45.0]     0.682129     2
0     2007-01-02  winter      44.1734  (40.0, 45.0]     0.660156     3
0     2007-01-02  winter      44.1734  (40.0, 45.0]     0.641602     4
...          ...     ...          ...           ...          ...   ...
2553  2013-12-29  winter      42.0134  (40.0, 45.0]     0.601562    19
2553  2013-12-29  winter      42.0134  (40.0, 45.0]     0.639648    20
2553  2013-12-29  winter      42.0134  (40.0, 45.0]     0.673828    21
2553  2013-12-29  winter      42.0134  (40.0, 45.0]     0.703613    22
2553  2013-12-29  winter      42.0134  (40.0, 45.0]     0.755371    23

[61296 rows x 6 columns]
